# Exp 04: Oversampling VIP animals

Card: [exp04_oversampling_vip.md](../../docs/experiments/exp04_oversampling_vip.md)


In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from core.training.config import ROOT
from core.training.config import VIP
from core.training.cv import CVResult
from core.training.runs import read_config
from core.analysis.gradcam import GradCamFigures
from core.training.tracking import start_run
from core.analysis.plots import save_figures, use_style

N_SHOW = 4
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.yaml"
PLAIN = CONFIG_DIR / "exp01_lecture_models" / "resnet18.yaml"  # trained in exp01, only read
FIGURES = ROOT / "docs" / "figures"

# dataviz reference palette, light mode; categorical slots in fixed order
MUTED = "#898781"
SERIES = {"plain (exp01)": "#eb6834", "oversampling x8": "#1baf7a"}
use_style()

## Training


In [ ]:
results = {
    "plain (exp01)": CVResult.load(PLAIN),
    "oversampling x8": CVResult.load(CONFIG),
}

## Training curves


In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, len(r.mean_curve(train_key)) + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(r.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="sampling", alignment="left")
fig.tight_layout()
figures = {"exp04_oversampling_vip_training.png": fig}
plt.show()

## AP comparison


In [ ]:
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["sampling"]
)
ap = fold_ap_df.groupby("sampling", sort=False).mean()
ap.round(3)

In [ ]:
# difference to exp01 counts only if it has same sign on both folds
diff = (
    results["oversampling x8"].fold_ap - results["plain (exp01)"].fold_ap
).T.add_prefix("diff fold ")
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"exp04_oversampling_vip_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "fast"}):
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})

save_figures(figures, FIGURES)